In [1]:
"""
Dataset and data loading utilities for addition task.
"""

import json
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path


class AdditionDataset(Dataset):
    """
    Dataset for multi-digit addition task.

    Loads pre-generated addition problems from JSON files.
    """

    def __init__(self, data_path, pad_token=0):
        """
        Initialize dataset.

        Args:
            data_path: Path to JSON data file
            pad_token: Token used for padding
        """
        self.pad_token = pad_token

        # Load data
        with open(data_path, 'r') as f:
            self.data = json.load(f)

        # Determine max lengths for padding
        self.max_input_len = max(len(s['input']) for s in self.data)
        self.max_target_len = max(len(s['target']) for s in self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """
        Get a single sample.

        Returns:
            dict containing:
                - input: Input sequence tensor
                - target: Target sequence tensor
                - input_len: Original input length
                - target_len: Original target length
        """
        sample = self.data[idx]

        # Convert to tensors
        input_seq = torch.tensor(sample['input'], dtype=torch.long)
        target_seq = torch.tensor(sample['target'], dtype=torch.long)

        # Store original lengths
        input_len = len(sample['input'])
        target_len = len(sample['target'])

        # Pad sequences to max length
        input_padded = torch.nn.functional.pad(
            input_seq,
            (0, self.max_input_len - input_len),
            value=self.pad_token
        )
        target_padded = torch.nn.functional.pad(
            target_seq,
            (0, self.max_target_len - target_len),
            value=self.pad_token
        )

        return {
            'input': input_padded,
            'target': target_padded,
            'input_len': input_len,
            'target_len': target_len
        }


def collate_fn(batch):
    """
    Custom collate function for batching.

    Args:
        batch: List of samples from dataset

    Returns:
        Dictionary with batched tensors
    """
    # Stack all tensors
    inputs = torch.stack([s['input'] for s in batch])
    targets = torch.stack([s['target'] for s in batch])
    input_lens = torch.tensor([s['input_len'] for s in batch])
    target_lens = torch.tensor([s['target_len'] for s in batch])

    return {
        'input': inputs,
        'target': targets,
        'input_len': input_lens,
        'target_len': target_lens
    }


def create_dataloaders(data_dir, batch_size=32, num_workers=0):
    """
    Create train, validation, and test dataloaders.

    Args:
        data_dir: Directory containing train.json, val.json, test.json
        batch_size: Batch size for dataloaders
        num_workers: Number of worker processes

    Returns:
        train_loader, val_loader, test_loader
    """
    data_dir = Path(data_dir)

    # Create datasets
    train_dataset = AdditionDataset(data_dir / 'train.json')
    val_dataset = AdditionDataset(data_dir / 'val.json')
    test_dataset = AdditionDataset(data_dir / 'test.json')

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=num_workers
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=num_workers
    )

    return train_loader, val_loader, test_loader


def get_vocab_size(num_digits=3):
    """
    Calculate vocabulary size for addition task.

    Args:
        num_digits: Number of digits in each operand

    Returns:
        Vocabulary size (digits 0-9 + operator + padding)
    """
    return 12  # 0-9 digits + operator token + padding token

In [2]:
"""
Attention mechanisms for sequence-to-sequence modeling.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.

    Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

    Args:
        Q: Query tensor [batch, ..., seq_len_q, d_k]
        K: Key tensor [batch, ..., seq_len_k, d_k]
        V: Value tensor [batch, ..., seq_len_v, d_k]
        mask: Optional mask [batch, ..., seq_len_q, seq_len_k]
              Values: 1 for positions to attend, 0 for positions to mask

    Returns:
        output: Attention output [batch, ..., seq_len_q, d_k]
        attention_weights: Attention weights [batch, ..., seq_len_q, seq_len_k]
    """
    d_k = Q.size(-1)

    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Scale scores
    scores = scores / math.sqrt(d_k)

    # Apply mask if provided (use masked_fill to set masked positions to -inf)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)

    # Apply attention to values
    output = torch.matmul(attention_weights, V)

    return output, attention_weights


class MultiHeadAttention(nn.Module):
    """
    Multi-head attention mechanism.

    Splits d_model into num_heads, applies attention in parallel,
    then concatenates and projects the results.
    """

    def __init__(self, d_model, num_heads):
        """
        Initialize multi-head attention.

        Args:
            d_model: Model dimension (must be divisible by num_heads)
            num_heads: Number of attention heads
        """
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Initialize linear projections for Q, K, V
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        # Initialize output projection
        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        Split tensor into multiple heads.

        Args:
            x: Input tensor [batch, seq_len, d_model]

        Returns:
            Tensor with shape [batch, num_heads, seq_len, d_k]
        """
        batch_size, seq_len, _ = x.size()

        # Reshape and transpose to split heads
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        x = x.transpose(1, 2)  # [batch, num_heads, seq_len, d_k]   
        return x

    def combine_heads(self, x):
        """
        Combine multiple heads back into single tensor.

        Args:
            x: Input tensor [batch, num_heads, seq_len, d_k]

        Returns:
            Tensor with shape [batch, seq_len, d_model]
        """
        batch_size, _, seq_len, d_k = x.size()

        # Transpose and reshape to combine heads
        x = x.transpose(1, 2)  # [batch, seq_len, num_heads, d_k]
        x = x.contiguous().view(batch_size, seq_len, self.d_model)
        return x


    def forward(self, query, key, value, mask=None):
        """
        Forward pass of multi-head attention.

        Args:
            query: Query tensor [batch, seq_len_q, d_model]
            key: Key tensor [batch, seq_len_k, d_model]
            value: Value tensor [batch, seq_len_v, d_model]
            mask: Optional attention mask

        Returns:
            output: Attention output [batch, seq_len_q, d_model]
            attention_weights: Attention weights [batch, num_heads, seq_len_q, seq_len_k]
        """
        batch_size = query.size(0)

        # Linear projections
        Q = self.W_Q(query)  # [batch, seq_len_q, d_model]
        K = self.W_K(key)    # [batch, seq_len_k, d_model]
        V = self.W_V(value)  # [batch, seq_len_v, d_model

        # Split heads
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # Apply scaled dot-product attention
        attention_output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads
        attention_output = self.combine_heads(attention_output)

        # Apply output projection
        output = self.W_O(attention_output)

        return output, attention_weights



def create_causal_mask(seq_len, device=None):
    """
    Create causal mask to prevent attending to future positions.

    Args:
        seq_len: Sequence length
        device: Device to create tensor on

    Returns:
        Mask tensor [1, 1, seq_len, seq_len] lower triangular matrix
    """
    # Lower triangular matrix
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)

In [4]:
"""
Sequence-to-sequence transformer model for addition task.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

from attention import MultiHeadAttention, create_causal_mask


class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.

    PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1) # [max_len, 1]
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)) # [d_model/2]
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        # Register as buffer (not a parameter)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """Add positional encoding to input embeddings."""
        # Add positional encoding to x
        x = x + self.pe[:, :x.size(1), :]
        return x
        


class FeedForward(nn.Module):
    """
    Position-wise feed-forward network.

    FFN(x) = max(0, xW1 + b1)W2 + b2
    """

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        # Initialize two linear layers and dropout
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)    

    def forward(self, x):
        """
        Apply feed-forward network.

        Args:
            x: Input tensor [batch, seq_len, d_model]

        Returns:
            Output tensor [batch, seq_len, d_model]
        """
        # Implement feed-forward with ReLU activation
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x
        


class EncoderLayer(nn.Module):
    """
    Single transformer encoder layer.

    Consists of multi-head self-attention and position-wise feed-forward,
    with residual connections and layer normalization.
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # Initialize components
        # - self-attention
        # - feed-forward
        # - layer normalizations
        # - dropout layers
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)


    def forward(self, x, mask=None):
        """
        Forward pass of encoder layer.

        Args:
            x: Input tensor [batch, seq_len, d_model]
            mask: Optional attention mask

        Returns:
            Output tensor [batch, seq_len, d_model]
        """
        # Self-attention with residual
        # Layer norm
        attn_output = self.self_attn(x, x, x, mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        
        # Feed-forward with residual
        # Layer norm
        ff_output = self.feed_forward(x)
        x = x + self.dropout2(ff_output)
        x = self.norm2(x)
        return x
        




class DecoderLayer(nn.Module):
    """
    Single transformer decoder layer.

    Consists of masked self-attention, encoder-decoder attention,
    and position-wise feed-forward, with residual connections and layer normalization.
    """

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # Initialize components
        # - masked self-attention
        # - encoder-decoder cross-attention
        # - feed-forward
        # - layer normalizations
        # - dropout layers
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Forward pass of decoder layer.

        Args:
            x: Decoder input [batch, tgt_len, d_model]
            encoder_output: Encoder output [batch, src_len, d_model]
            src_mask: Source mask for cross-attention
            tgt_mask: Target mask for self-attention (causal)

        Returns:
            Output tensor [batch, tgt_len, d_model]
        """
        # Masked self-attention with residual
        # Layer norm
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)

        # Cross-attention with residual
        # Layer norm
        attn_output = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = x + self.dropout2(attn_output)
        x = self.norm2(x)

        # Feed-forward with residual
        # Layer norm
        ff_output = self.feed_forward(x)
        x = x + self.dropout3(ff_output)
        x = self.norm3(x)
        return x



class Seq2SeqTransformer(nn.Module):
    """
    Full sequence-to-sequence transformer model.
    """

    def __init__(
        self,
        vocab_size,
        d_model=128,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=512,
        dropout=0.1,
        max_len=100
    ):
        super().__init__()

        self.d_model = d_model
        self.vocab_size = vocab_size

        # Embeddings
        self.encoder_embedding = nn.Embedding(vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(vocab_size, d_model)

        # Positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len)

        # Initialize encoder layers
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])

        # Initialize decoder layers
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])

        # Output projection
        self.output_projection = nn.Linear(d_model, vocab_size)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize model weights."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        """
        Encode source sequence.

        Args:
            src: Source tokens [batch, src_len]
            src_mask: Source padding mask

        Returns:
            Encoder output [batch, src_len, d_model]
        """
        # Embed and scale
        x = self.encoder_embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        # Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x, src_mask)

        return x    

    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        """
        Decode target sequence given encoder output.

        Args:
            tgt: Target tokens [batch, tgt_len]
            encoder_output: Encoder output [batch, src_len, d_model]
            src_mask: Source padding mask
            tgt_mask: Target causal mask

        Returns:
            Decoder output [batch, tgt_len, d_model]
        """
        # Embed and scale
        x = self.decoder_embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)

        # Pass through decoder layers
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        return x    


    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        Full forward pass.

        Args:
            src: Source tokens [batch, src_len]
            tgt: Target tokens [batch, tgt_len]
            src_mask: Source padding mask
            tgt_mask: Target causal mask

        Returns:
            Output logits [batch, tgt_len, vocab_size]
        """
        # Encode source
        # Decode target
        # Project to vocabulary

        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        output_logits = self.output_projection(decoder_output)  

        return output_logits

    def generate(self, src, max_len=20, start_token=0):
        """
        Generate output sequence using greedy decoding.

        Args:
            src: Source tokens [batch, src_len]
            max_len: Maximum generation length
            start_token: Start-of-sequence token

        Returns:
            Generated tokens [batch, max_len]
        """
        self.eval()
        device = src.device
        batch_size = src.size(0)

        with torch.no_grad():
            # Encode source once
            encoder_output = self.encode(src)

            # Initialize target with start token
            tgt = torch.full((batch_size, 1), start_token, dtype=torch.long, device=device)

            for _ in range(max_len - 1):
                # Create causal mask for current length
                tgt_mask = create_causal_mask(tgt.size(1), device=device)

                # Decode current sequence
                decoder_output = self.decode(tgt, encoder_output, tgt_mask=tgt_mask)

                # Get next token predictions
                logits = self.output_projection(decoder_output[:, -1, :])
                next_token = logits.argmax(dim=-1, keepdim=True)

                # Append to target sequence
                tgt = torch.cat([tgt, next_token], dim=1)

        return tgt

    def get_attention_weights(self, src, tgt):
        """
        Run a forward pass and return attention weights.
        """
        src_mask = torch.zeros(src.shape[0], src.shape[1], device=self.device).bool()
        tgt_mask = create_causal_mask(tgt.shape[1]).to(self.device)
        memory_mask = None

        encoder_attentions = []
        decoder_self_attentions = []
        decoder_cross_attentions = []

        hooks = []

        def make_hook(attention_list):
            def hook(module, input, output):
                attention_list.append(output[1].detach())
            return hook

        # Register hooks
        for layer in self.encoder_layers:
            hooks.append(layer.self_attn.register_forward_hook(make_hook(encoder_attentions)))
        for layer in self.decoder_layers:
            hooks.append(layer.self_attn.register_forward_hook(make_hook(decoder_self_attentions)))
            hooks.append(layer.cross_attn.register_forward_hook(make_hook(decoder_cross_attentions)))

        # Forward pass
        self.forward(src, tgt, tgt_mask=tgt_mask)

        # Remove hooks
        for h in hooks:
            h.remove()

        return {
            'encoder_attention': torch.cat(encoder_attentions, dim=0),
            'decoder_self_attention': torch.cat(decoder_self_attentions, dim=0),
            'decoder_cross_attention': torch.cat(decoder_cross_attentions, dim=0)
        }    

In [ ]:
"""
Training script for sequence-to-sequence addition model.
"""

import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
import json
import argparse
from tqdm import tqdm
import time

from model import Seq2SeqTransformer
from dataset import create_dataloaders, get_vocab_size
from attention import create_causal_mask


def compute_accuracy(outputs, targets, pad_token=0):
    """
    Compute sequence-level accuracy.

    Args:
        outputs: Model predictions [batch, seq_len, vocab_size]
        targets: Ground truth [batch, seq_len]
        pad_token: Padding token to ignore

    Returns:
        Accuracy (fraction of completely correct sequences)
    """
    # Get predicted tokens from logits
    preds = outputs.argmax(dim=-1)  # [batch, seq_len]

    # Create mask for non-padding positions
    mask = (targets != pad_token)  # [batch, seq_len]

    # Check if entire sequence matches (excluding padding)
    correct = ((preds == targets) | ~mask).all(dim=1)  # [batch]
    accuracy = correct.float().mean().item()
    return accuracy

    


def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Train for one epoch.

    Args:
        model: Transformer model
        dataloader: Training dataloader
        criterion: Loss function
        optimizer: Optimizer
        device: Device to run on

    Returns:
        Average loss, average accuracy
    """
    model.train()
    total_loss = 0
    total_acc = 0
    num_batches = 0

    progress = tqdm(dataloader, desc="Training")
    for batch in progress:
        # Move to device
        inputs = batch['input'].to(device)
        targets = batch['target'].to(device)

        # Prepare decoder input and output for teacher forcing
        # Decoder input should be targets shifted right (exclude last token)
        # Decoder output should be targets shifted left (exclude first token)
        decoder_input = targets[:, :-1]
        decoder_output = targets[:, 1:] 


        # Create causal mask for decoder (using shifted sequence length)
        seq_len = decoder_input.size(1)
        causal_mask = create_causal_mask(seq_len).to(device)

        # Forward pass
        outputs = model(
            src=inputs,
            tgt=decoder_input,
            tgt_mask=causal_mask
        )  # [batch, seq_len, vocab_size]

        # Compute loss
        # Hint: Flatten for cross entropy - need 2D tensors
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            decoder_output.contiguous().view(-1)
        )

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Compute accuracy
        acc = compute_accuracy(outputs, decoder_output)

        # Update progress bar
        progress.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{acc:.2%}'
        })

        total_loss += loss.item()
        total_acc += acc
        num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def evaluate(model, dataloader, criterion, device):
    """
    Evaluate model on validation/test set.

    Args:
        model: Transformer model
        dataloader: Evaluation dataloader
        criterion: Loss function
        device: Device to run on

    Returns:
        Average loss, average accuracy
    """
    model.eval()
    total_loss = 0
    total_acc = 0
    num_batches = 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)

            # Prepare decoder input and output (same as training)
            decoder_input = targets[:, :-1]
            decoder_output = targets[:, 1:]

            # Create causal mask (using shifted sequence length)   
            seq_len = decoder_input.size(1)
            causal_mask = create_causal_mask(seq_len).to(device)

            # Forward pass
            outputs = model(
                src=inputs,
                tgt=decoder_input,
                tgt_mask=causal_mask
            )  # [batch, seq_len, vocab_size]

            # Compute loss and accuracy (flatten for cross entropy)
            loss = criterion(
                outputs.view(-1, outputs.size(-1)),
                decoder_output.contiguous().view(-1)
            )
            acc = compute_accuracy(outputs, decoder_output)
            # Accumulate metrics

            total_loss += loss.item()
            total_acc += acc
            num_batches += 1

    return total_loss / num_batches, total_acc / num_batches


def main():
    parser = argparse.ArgumentParser(description='Train addition transformer')
    parser.add_argument('--data-dir', default='data', help='Data directory')
    parser.add_argument('--output-dir', default='results', help='Output directory')
    parser.add_argument('--batch-size', type=int, default=64, help='Batch size')
    parser.add_argument('--epochs', type=int, default=50, help='Number of epochs')
    parser.add_argument('--lr', type=float, default=1e-3, help='Learning rate')
    parser.add_argument('--d-model', type=int, default=128, help='Model dimension')
    parser.add_argument('--num-heads', type=int, default=4, help='Number of attention heads')
    parser.add_argument('--num-layers', type=int, default=2, help='Number of encoder/decoder layers')
    parser.add_argument('--d-ff', type=int, default=512, help='Feed-forward dimension')
    parser.add_argument('--dropout', type=float, default=0.1, help='Dropout rate')
    parser.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')
    parser.add_argument('--seed', type=int, default=42, help='Random seed')

    args = parser.parse_args()

    # Set random seed
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)

    # Create output directory
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Load data
    train_loader, val_loader, test_loader = create_dataloaders(
        args.data_dir, args.batch_size
    )

    # Create model
    vocab_size = get_vocab_size()
    model = Seq2SeqTransformer(
        vocab_size=vocab_size,
        d_model=args.d_model,
        num_heads=args.num_heads,
        num_encoder_layers=args.num_layers,
        num_decoder_layers=args.num_layers,
        d_ff=args.d_ff,
        dropout=args.dropout
    ).to(args.device)

    # Initialize optimizer (Adam recommended)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)

    # Initialize learning rate scheduler (ReduceLROnPlateau recommended)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # Initialize loss function (use nn.CrossEntropyLoss)
    criterion = nn.CrossEntropyLoss(ignore_index=0)  # Assuming 0 is the padding token

    # Training loop
    best_val_acc = -1
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    print(f"Starting training for {args.epochs} epochs...")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")

        # Train
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, args.device
        )

        # Validate
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, args.device
        )

        # Step learning rate scheduler (pass val_loss)
        scheduler.step(val_loss)
        

        # Log results
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2%}")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2%}")

        training_history['train_loss'].append(train_loss)
        training_history['train_acc'].append(train_acc)
        training_history['val_loss'].append(val_loss)
        training_history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), output_dir / 'best_model.pth')
            print(f"Saved best model with validation accuracy: {val_acc:.2%}")

    # Test final model
    model.load_state_dict(torch.load(output_dir / 'best_model.pth'))
    test_loss, test_acc = evaluate(model, test_loader, criterion, args.device)
    print(f"\nTest Loss: {test_loss:.4f}, Test Acc: {test_acc:.2%}")

    # Save training history
    training_history['test_loss'] = test_loss
    training_history['test_acc'] = test_acc
    with open(output_dir / 'training_log.json', 'w') as f:
        json.dump(training_history, f, indent=2)

    print(f"\nTraining complete! Results saved to {output_dir}")


if __name__ == '__main__':
    main()

In [ ]:
"""
Analysis and visualization of attention patterns.
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import argparse
from tqdm import tqdm

from model import Seq2SeqTransformer
from dataset import create_dataloaders, get_vocab_size
from attention import create_causal_mask


def extract_attention_weights(model, dataloader, device, num_samples=100):
    """
    Extract attention weights from model for analysis.

    Args:
        model: Trained transformer model
        dataloader: Data loader
        device: Device to run on
        num_samples: Number of samples to analyze

    Returns:
        Dictionary containing attention weights and sample data
    """
    model.eval()

    all_encoder_attentions = []
    all_decoder_self_attentions = []
    all_decoder_cross_attentions = []
    all_inputs = []
    all_targets = []

    samples_collected = 0

    with torch.no_grad():
        for batch in dataloader:
            if samples_collected >= num_samples:
                break

            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)
            batch_size = inputs.size(0)

            # Modify model forward pass to return attention weights
            # This requires updating the model to store/return attention weights
            




            # For now, we'll need to hook into the attention layers
            encoder_attentions = []
            decoder_self_attentions = []
            decoder_cross_attentions = []

            # Register hooks to capture attention weights
            def make_hook(attention_list):
                def hook(module, input, output):
                    # output is (attention_output, attention_weights)
                    attention_list.append(output[1].detach().cpu())
                return hook

            # Register hooks on attention layers
            # You'll need to access model.encoder_layers[i].self_attn
            # and model.decoder_layers[i].self_attn, cross_attn
            hooks = []
            for layer in model.encoder_layers:
                h = layer.self_attn.register_forward_hook(
                    make_hook(encoder_attentions)
                )
                hooks.append(h)
            for layer in model.decoder_layers:
                h1 = layer.self_attn.register_forward_hook(
                    make_hook(decoder_self_attentions)
                )
                h2 = layer.cross_attn.register_forward_hook(
                    make_hook(decoder_cross_attentions)
                )
                hooks.extend([h1, h2])

            # Forward pass
            # Run model forward pass
            outputs = model(
                src=inputs,
                tgt=targets[:, :-1],
                tgt_mask=create_causal_mask(targets.size(1) - 1).to(device)
            )

            # Collect samples
            samples_to_take = min(batch_size, num_samples - samples_collected)
            all_inputs.extend(inputs[:samples_to_take].cpu().numpy())
            all_targets.extend(targets[:samples_to_take].cpu().numpy())

            # Collect attention weights from hooks
            for i in range(len(encoder_attentions)):
                all_encoder_attentions.append(
                    encoder_attentions[i][:samples_to_take].numpy()
                )
            for i in range(len(decoder_self_attentions)):
                all_decoder_self_attentions.append(
                    decoder_self_attentions[i][:samples_to_take].numpy()
                )
            for i in range(len(decoder_cross_attentions)):
                all_decoder_cross_attentions.append(
                    decoder_cross_attentions[i][:samples_to_take].numpy()
                )

            samples_collected += samples_to_take

    return {
        'encoder_attention': all_encoder_attentions,
        'decoder_self_attention': all_decoder_self_attentions,
        'decoder_cross_attention': all_decoder_cross_attentions,
        'inputs': all_inputs,
        'targets': all_targets
    }


def visualize_attention_pattern(attention_weights, input_tokens, output_tokens,
                               title="Attention Pattern", save_path=None):
    """
    Visualize attention weights as heatmap.

    Args:
        attention_weights: Attention weights [num_heads, out_len, in_len]
        input_tokens: Input token labels
        output_tokens: Output token labels
        title: Plot title
        save_path: Path to save figure
    """
    num_heads = attention_weights.shape[0]

    # Create figure with subplots for each head
    fig, axes = plt.subplots(
        2, (num_heads + 1) // 2,
        figsize=(5 * ((num_heads + 1) // 2), 8)
    )
    axes = axes.flatten()

    for head_idx in range(num_heads):
        ax = axes[head_idx]

        # Plot heatmap
        sns.heatmap(
            attention_weights[head_idx],
            ax=ax,
            cmap='Blues',
            cbar=True,
            square=True,
            xticklabels=input_tokens,
            yticklabels=output_tokens,
            vmin=0,
            vmax=1
        )

        ax.set_title(f'Head {head_idx + 1}')
        ax.set_xlabel('Input Position')
        ax.set_ylabel('Output Position')

    # Hide unused subplots
    for idx in range(num_heads, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def analyze_head_specialization(attention_data, output_dir, num_digits):
    """
    Analyze what each attention head specializes in.

    Args:
        attention_data: Dictionary with attention weights and samples
        output_dir: Directory to save analysis results
        num_digits: Number of digits in the addition problem
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Analyze encoder self-attention
    print("Analyzing encoder self-attention patterns...")

    # For each head, compute statistics:
    # - Average attention to operator token
    # - Average attention to same position (diagonal)
    # - Average attention to carry positions
    # - Entropy of attention distribution

    head_stats = {}

    # Implement analysis
    for head_idx in range(len(attention_data['encoder_attention'])):
        # Extract attention weights for this head
        head_attention_weights = np.array([
            sample[head_idx] for sample in attention_data['encoder_attention']
        ])  # Shape: [num_samples, out_len, in_len]

        # Define indices for analysis
        operator_token_idx = num_digits

        # Define carry positions: pairs of corresponding digit indices
        # E.g., for 3 digits: (2, 6), (1, 5), (0, 4)
        # We look at attention from num1's digits to num2's digits and vice-versa
        carry_attention_values = []
        for i in range(num_digits):
            # Attention from i-th digit of num1 to i-th digit of num2
            # Digit order is reversed in input, so we use num_digits - 1 - i
            pos1 = num_digits - 1 - i
            pos2 = 2 * num_digits - i
            carry_attention_values.append(head_attention_weights[:, pos1, pos2])
            carry_attention_values.append(head_attention_weights[:, pos2, pos1])
        
        avg_attention_to_carry_positions = np.mean(carry_attention_values)


        # Compute average attention to the operator token
        avg_attention_to_operator = head_attention_weights[:, :, operator_token_idx].mean()

        # Compute average attention to the same position (diagonal)
        avg_attention_to_same_position = np.diagonal(head_attention_weights, axis1=-2, axis2=-1).mean()

        entropy = -np.sum(head_attention_weights * np.log(head_attention_weights + 1e-10), axis=-1).mean()

        head_stats[head_idx] = {
            'avg_attention_to_operator': avg_attention_to_operator,
            'avg_attention_to_same_position': avg_attention_to_same_position,
            'avg_attention_to_carry_positions': avg_attention_to_carry_positions,
            'entropy': entropy
        }

    # Save analysis results
    with open(output_dir / 'head_analysis.json', 'w') as f:
        json.dump(head_stats, f, indent=2)

    return head_stats


def ablation_study(model, dataloader, device, output_dir):
    """
    Perform head ablation study.

    Test model performance when individual heads are disabled.

    Args:
        model: Trained model
        dataloader: Test dataloader
        device: Device to run on
        output_dir: Directory to save results
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Running head ablation study...")

    # Get baseline accuracy
    baseline_acc = evaluate_model(model, dataloader, device)
    print(f"Baseline accuracy: {baseline_acc:.2%}")

    ablation_results = {'baseline': baseline_acc}

    # TODO: For each layer and head:
    # 1. Temporarily zero out the head's output
    # 2. Evaluate model performance
    # 3. Restore the head
    # 4. Record the performance drop
    for layer_idx, layer in enumerate(model.encoder_layers):
        num_heads = layer.self_attn.num_heads
        for head_idx in range(num_heads):
            print(f"Ablating Layer {layer_idx} Head {head_idx}...")

            # Store original projection weights
            original_W_O = layer.self_attn.W_O.weight.data.clone()
            original_W_O_bias = layer.self_attn.W_O.bias.data.clone()

            # Zero out the head's output by modifying W_O
            with torch.no_grad():
                d_k = layer.self_attn.d_k
                start = head_idx * d_k
                end = (head_idx + 1) * d_k
                layer.self_attn.W_O.weight.data[:, start:end] = 0.0
                layer.self_attn.W_O.bias.data += 0.0  # No change to bias

            # Evaluate model
            ablated_acc = evaluate_model(model, dataloader, device)
            print(f"  Ablated accuracy: {ablated_acc:.2%}")

            # Restore original weights
            with torch.no_grad():
                layer.self_attn.W_O.weight.data = original_W_O
                layer.self_attn.W_O.bias.data = original_W_O_bias

            # Record results
            ablation_results[f'layer_{layer_idx}_head_{head_idx}'] = ablated_acc

    # Save ablation results
    with open(output_dir / 'ablation_results.json', 'w') as f:
        json.dump(ablation_results, f, indent=2)

    # Create visualization of head importance
    plot_head_importance(ablation_results, output_dir / 'head_importance.png')

    return ablation_results


def evaluate_model(model, dataloader, device):
    """
    Evaluate model accuracy.

    Args:
        model: Model to evaluate
        dataloader: Test dataloader
        device: Device to run on

    Returns:
        Accuracy
    """
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)

            # Generate predictions
            # Compare with targets
            # Count correct sequences
            predictions = model.generate(inputs, max_len=targets.size(1))
            for pred, target in zip(predictions, targets):
                if torch.equal(pred, target):
                    correct += 1
                total += 1


    return correct / total


def plot_head_importance(ablation_results, save_path):
    """
    Visualize head importance from ablation study.

    Args:
        ablation_results: Dictionary of ablation results
        save_path: Path to save figure
    """
    # Extract performance drops for each head
    baseline = ablation_results['baseline']

    # Create bar plot showing accuracy drop when each head is removed

    head_labels = []
    accuracy_drops = []
    for key, acc in ablation_results.items():
        if key == 'baseline':
            continue
        head_labels.append(key)
        accuracy_drops.append(baseline - acc)


   

    plt.figure(figsize=(12, 6))

    # Plot bars for each head
    plt.bar(head_labels, accuracy_drops, color='skyblue')
    plt.axhline(0, color='gray', linestyle='--')
    plt.title('Head Importance (Accuracy Drop When Removed)')
    plt.ylabel('Accuracy Drop')
    plt.xticks(rotation=45)


    plt.xlabel('Head')
    plt.ylabel('Accuracy Drop')
    plt.title('Head Importance (Accuracy Drop When Removed)')
    plt.xticks(rotation=45)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def visualize_example_predictions(model, dataloader, device, output_dir, num_examples=5):
    """
    Visualize model predictions on example inputs.

    Args:
        model: Trained model
        dataloader: Data loader
        device: Device to run on
        output_dir: Directory to save visualizations
        num_examples: Number of examples to visualize
    """
    output_dir = Path(output_dir)
    (output_dir / 'examples').mkdir(parents=True, exist_ok=True)

    model.eval()

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= num_examples:
                break

            inputs = batch['input'].to(device)
            targets = batch['target'].to(device)

            # Take first sample from batch
            input_seq = inputs[0:1]
            target_seq = targets[0]

            # Generate prediction
            # Use model.generate() to get prediction
            prediction = model.generate(input_seq, max_len=target_seq.size(0))

            # Convert to strings for visualization
            input_str = ' '.join(map(str, input_seq[0].cpu().numpy()))
            target_str = ''.join(map(str, target_seq.cpu().numpy()))
            pred_str = ''.join(map(str, prediction[0].cpu().numpy()))

            print(f"\nExample {batch_idx + 1}:")
            print(f"  Input:  {input_str}")
            print(f"  Target: {target_str}")
            print(f"  Pred:   {pred_str}")
            print(f"  Correct: {target_str == pred_str}")

            # Extract and visualize attention for this example
            attention_weights = model.get_attention_weights(input_seq, prediction.unsqueeze(0))

            # Visualize and save encoder self-attention
            visualize_attention_pattern(
                attention_weights['encoder_attention'].squeeze(0).cpu().numpy(),
                input_tokens=[str(i) for i in input_seq[0].cpu().numpy()],
                output_tokens=[str(i) for i in input_seq[0].cpu().numpy()],
                title=f"Example {batch_idx + 1}: Encoder Self-Attention",
                save_path=output_dir / 'examples' / f'example_{batch_idx}_encoder_self_attn.png'
            )

            # Visualize and save decoder cross-attention
            visualize_attention_pattern(
                attention_weights['decoder_cross_attention'].squeeze(0).cpu().numpy(),
                input_tokens=[str(i) for i in input_seq[0].cpu().numpy()],
                output_tokens=[str(i) for i in prediction[0].cpu().numpy()],
                title=f"Example {batch_idx + 1}: Decoder Cross-Attention",
                save_path=output_dir / 'examples' / f'example_{batch_idx}_decoder_cross_attn.png'
            )
            



def main():
    parser = argparse.ArgumentParser(description='Analyze attention patterns')
    parser.add_argument('--model-path', required=True, help='Path to trained model')
    parser.add_argument('--data-dir', default='data', help='Data directory')
    parser.add_argument('--output-dir', default='results', help='Output directory')
    parser.add_argument('--batch-size', type=int, default=32, help='Batch size')
    parser.add_argument('--num-samples', type=int, default=100, help='Number of samples to analyze')
    parser.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')

    args = parser.parse_args()

    # Load model
    vocab_size = get_vocab_size()
    model = Seq2SeqTransformer(
        vocab_size=vocab_size,
        d_model=128,
        num_heads=4,
        num_encoder_layers=2,
        num_decoder_layers=2,
        d_ff=512
    ).to(args.device)

    model.load_state_dict(torch.load(args.model_path))
    print(f"Loaded model from {args.model_path}")

    # Load data
    _, _, test_loader = create_dataloaders(args.data_dir, args.batch_size)

    # Create output directories
    output_dir = Path(args.output_dir)
    (output_dir / 'attention_patterns').mkdir(parents=True, exist_ok=True)
    (output_dir / 'head_analysis').mkdir(parents=True, exist_ok=True)

    # Extract attention weights
    print("Extracting attention weights...")
    attention_data = extract_attention_weights(
        model, test_loader, args.device, args.num_samples
    )

    # Analyze head specialization
    head_stats = analyze_head_specialization(
        attention_data, output_dir / 'head_analysis', args.num_digits
    )

    # Run ablation study
    ablation_results = ablation_study(
        model, test_loader, args.device, output_dir / 'head_analysis'
    )

    # Visualize example predictions
    visualize_example_predictions(
        model, test_loader, args.device, output_dir, num_examples=5
    )

    print(f"\nAnalysis complete! Results saved to {output_dir}")


if __name__ == '__main__':
    main()